# AVO and Gassmann: testing a DHI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_avo_gassmann_dhi.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 6.3 Zoeppritz and Shuey
2. 6.5 Synthetic AVO Gathers
3. 6.6 Acoustic or Elastic?

In [ ]:
# Ask Ogbon starter: Test a direct hydrocarbon indicator
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
FLUID = 'gas'
MAX_ANGLE_DEG = 35
# brine sand baseline (typical values; replace with yours)
VP_SH, VS_SH, RHO_SH = 3050.0, 1400.0, 2.40   # shale cap
VP_BR, VS_BR, RHO_BR = 2650.0, 1350.0, 2.25   # brine sand
PHI = 0.25          # porosity
K_MIN = 36.0        # mineral modulus (GPa, quartz)
K_FL_BRINE = 2.6    # brine modulus (GPa)
K_FL_NEW = 0.05      # target fluid modulus (GPa)
RHO_FL_BRINE, RHO_FL_NEW = 1.05, 0.2

In [ ]:
def gassmann_ksat(k_dry, k_min, k_fl, phi):
    num = (1 - k_dry/k_min) ** 2
    den = phi/k_fl + (1 - phi)/k_min - k_dry/k_min**2
    return k_dry + num / den

def gassmann_kdry(k_sat, k_min, k_fl, phi):
    num = k_sat * (phi*k_min/k_fl + 1 - phi) - k_min
    den = phi*k_min/k_fl + k_sat/k_min - 1 - phi
    return num / den

def shuey_rpp(vp1, vs1, r1, vp2, vs2, r2, theta_deg):
    th = np.radians(theta_deg)
    vp, vs, rho = (vp1+vp2)/2, (vs1+vs2)/2, (r1+r2)/2
    dvp, dvs, dr = vp2-vp1, vs2-vs1, r2-r1
    r0 = 0.5 * (dvp/vp + dr/rho)
    g = 0.5*dvp/vp - 2*(vs/vp)**2 * (dr/rho + 2*dvs/vs)
    f = 0.5 * dvp/vp
    return r0 + g*np.sin(th)**2 + f*(np.tan(th)**2 - np.sin(th)**2)

# 1) moduli of the brine sand from its velocities (GPa, g/cc, m/s)
mu = RHO_BR * VS_BR**2 * 1e-6
k_sat_br = RHO_BR * (VP_BR**2 - 4.0/3.0 * VS_BR**2) * 1e-6

# 2) strip the brine out, put the new fluid in
k_dry = gassmann_kdry(k_sat_br, K_MIN, K_FL_BRINE, PHI)
k_sat2 = gassmann_ksat(k_dry, K_MIN, K_FL_NEW, PHI)
rho2 = RHO_BR + PHI * (RHO_FL_NEW - RHO_FL_BRINE)
vs2 = np.sqrt(mu * 1e6 / rho2)
vp2 = np.sqrt((k_sat2 + 4.0/3.0 * mu) * 1e6 / rho2)
print('substituted sand: Vp %.0f  Vs %.0f  rho %.2f' % (vp2, vs2, rho2))

# 3) the AVO curves: brine baseline vs the substituted sand
th = np.linspace(0, MAX_ANGLE_DEG, 60)
r_br = shuey_rpp(VP_SH, VS_SH, RHO_SH, VP_BR, VS_BR, RHO_BR, th)
r_hc = shuey_rpp(VP_SH, VS_SH, RHO_SH, vp2, vs2, rho2, th)

plt.figure(figsize=(6, 4))
plt.plot(th, r_br, label='brine baseline')
plt.plot(th, r_hc, label=FLUID + ' substituted')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('incidence angle (deg)'); plt.ylabel('Rpp')
plt.title('the deviation from brine IS the fluid signal')
plt.legend(); plt.tight_layout(); plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)